In [1]:

# Taken from Medium article: https://imlamrin.medium.com/detecting-credit-card-fraud-with-autoencoders-using-pytorch-from-scratch-89715e769193

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd

# Load dataset
data = pd.read_csv("../data/creditcard.csv")  # Make sure the dataset is in the same directory or provide the full path

# Check for missing values
print(data.isnull().sum())  # Ensure there are no missing values

# Separate features and labels
features = data.drop('Class', axis=1).values  # Drop the 'Class' column for features
labels = data['Class'].values  # 'Class' is the target variable

# Standardize the features (this is important for training neural networks)
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.FloatTensor(y_train)
y_test = torch.FloatTensor(y_test)

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

input_dim = X_train.shape[1]
autoencoder = Autoencoder(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)

num_epochs = 50
batch_size = 128

for epoch in range(num_epochs):
    for i in range(0, len(X_train), batch_size):
        batch = X_train[i:i+batch_size]
        output = autoencoder(batch)
        loss = criterion(output, batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


with torch.no_grad():
    reconstructions = autoencoder(X_test)
    reconstruction_error = torch.mean((reconstructions - X_test) ** 2, axis=1)

# Set a threshold for classification
threshold = np.percentile(reconstruction_error.numpy(), 95)  # 95th percentile

# Classify based on reconstruction error
y_pred = (reconstruction_error.numpy() > threshold).astype(int)

# Evaluate
print(classification_report(y_test, y_pred))

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64
Epoch [1/50], Loss: 0.0146
Epoch [2/50], Loss: 0.0071
Epoch [3/50], Loss: 0.0039
Epoch [4/50], Loss: 0.0025
Epoch [5/50], Loss: 0.0015
Epoch [6/50], Loss: 0.0015
Epoch [7/50], Loss: 0.0015
Epoch [8/50], Loss: 0.0012
Epoch [9/50], Loss: 0.0008
Epoch [10/50], Loss: 0.0013
Epoch [11/50], Loss: 0.0008
Epoch [12/50], Loss: 0.0009
Epoch [13/50], Loss: 0.0007
Epoch [14/50], Loss: 0.0006
Epoch [15/50], Loss: 0.0006
Epoch [16/50], Loss: 0.0005
Epoch [17/50], Loss: 0.0005
Epoch [18/50], Loss: 0.0008
Epoch [19/50], Loss: 0.0008
Epoch [20/50], Loss: 0.0006
Epoch [21/50], Loss: 0.0004
Epoch [22/50], Loss: 0.0004
Epoch [2